In [1]:
import numpy as np
from netCDF4 import Dataset
import matplotlib.pyplot as plt
#import cartopy.crs as crs
from cartopy.io.shapereader import Reader
from cartopy.feature import ShapelyFeature
from cartopy.feature import NaturalEarthFeature
import cartopy.crs as ccrs
from datetime import datetime, timedelta
from metpy.units import units
from metpy.calc import dewpoint_from_specific_humidity, relative_humidity_from_specific_humidity, wind_speed, specific_humidity_from_mixing_ratio
import pyart
from wrf import getvar
import haversine
from metpy.interpolate import log_interpolate_1d, interpolate_1d


## You are using the Python ARM Radar Toolkit (Py-ART), an open source
## library for working with weather radar data. Py-ART is partly
## supported by the U.S. Department of Energy as part of the Atmospheric
## Radiation Measurement (ARM) Climate Research Facility, an Office of
## Science user facility.
##
## If you use this software to prepare a publication, please cite:
##
##     JJ Helmus and SM Collis, JORS 2016, doi: 10.5334/jors.119



In [2]:
wspd_thresh = 20 #wind speed cutoff for UAS ops, in m/s

dt = datetime(2022,9,15,10)
#dt = datetime(2022,7,19,10)
#dt = datetime(2021,6,4,10)

# lons_newuas = [-84.55, -84.93, -84.55, -84.09, -84.15, -84.52, -84.94, -85.21, -85.29]
# lats_newuas = [39.21, 39.41, 39.48, 39.47, 38.92, 38.86, 38.95, 39.19, 38.86]
# lons_newuas = [-84.55]
# lats_newuas = [39.21]

lons_newuas = [-84.548065]
lats_newuas = [39.212013]

prof_lons = lons_newuas
prof_lats = lats_newuas

In [3]:
#Make arrays of heights and times
#For 1 km profile at a 3 m/s ascent/descent rate

#profile depth, in m
depth = 1000

#profile resolution, in m
res = 50

#uas_z = np.concatenate([np.arange(res, depth+res, res), np.arange(depth-res, 0, res*-1)])
#uas_z = np.concatenate([np.arange(res, depth+res, res)])
uas_z = np.concatenate([np.arange(res-res, depth+res, res)])

#Get time, in seconds, for this profile

#ascent rate (in m/s)
ar = 3

#uas_time = np.arange(res/ar, ((len(uas_z)*res)/ar)+(res/ar), res/ar) 
# uas_time = np.arange(0, ((len(uas_z)*res)/ar), res/ar) 
uas_time = np.arange(0, ((len(uas_z)*res)/ar), res/ar) 

print(uas_z)
print(uas_time)

print(uas_z.shape)
print(uas_time.shape)

#Replicate this profile hourly
#Interval between flights (in seconds)
fl_int = 3600
#Number of flights
fl_num = 10
#Start day
st_day = 154024
#st_day = 153966
#st_day = 153556

#first_time (in seconds)
st_sec = 3*fl_int
end_sec = (3+fl_num)*fl_int
#Get a range of times

time_range = np.arange((st_sec/86400), (end_sec/86400), fl_int/86400) + st_day

#Now get the times assuming launches at each interval

uas_times2=[]
uas_z2 = []
for time_l in time_range:
    time_prof = time_l + (uas_time/86400)
    uas_times2.append(time_prof)
    uas_z2.append(uas_z)
uas_times2 = np.concatenate(uas_times2)
uas_z2=np.concatenate(uas_z2)
print(uas_times2)

[   0   50  100  150  200  250  300  350  400  450  500  550  600  650
  700  750  800  850  900  950 1000]
[  0.          16.66666667  33.33333333  50.          66.66666667
  83.33333333 100.         116.66666667 133.33333333 150.
 166.66666667 183.33333333 200.         216.66666667 233.33333333
 250.         266.66666667 283.33333333 300.         316.66666667
 333.33333333]
(21,)
(21,)
[154024.125      154024.1251929  154024.1253858  154024.1255787
 154024.1257716  154024.12596451 154024.12615741 154024.12635031
 154024.12654321 154024.12673611 154024.12692901 154024.12712191
 154024.12731481 154024.12750772 154024.12770062 154024.12789352
 154024.12808642 154024.12827932 154024.12847222 154024.12866512
 154024.12885802 154024.16666667 154024.16685957 154024.16705247
 154024.16724537 154024.16743827 154024.16763117 154024.16782407
 154024.16801698 154024.16820988 154024.16840278 154024.16859568
 154024.16878858 154024.16898148 154024.16917438 154024.16936728
 154024.16956019 154024.1

In [4]:
#Obs types for the UAS obs. Change once we have this implemented in DART
otype_T = 123
otype_q = 120
#otype_q = 124
otype_u = 121
otype_v = 122

#Errors for UAS obs
oerr_T = 0.5
oerr_q = 0.5
#oerr_q = 0.001
oerr_u = 1.0
oerr_v = 1.0

otype_s = []
obs_s = []
lon_s = []
lat_s = []
elev_s = []
error_s = []
time_s = []
pres_s = []

#minute_range = np.arange(180,605,5)
minute_range = np.arange(180,190,5)

#minute_range = np.arange(595,605,5)

for mins in minute_range:
    dt_start = datetime(2022,9,15,0,0)
    #dt_start = datetime(2022,7,19,0,0)
    #dt_start = datetime(2021,6,4,0,0)
    
    dt = dt_start + timedelta(minutes=int(mins))
    print(dt)
    wrfout = Dataset('/glade/derecho/scratch/mawilson/20210604_newcase/nature_100IOP6/final_nature/wrfout_d02_2022-'+str(dt.isoformat()[5:7])+'-'+str(dt.isoformat()[8:10])+'_'+str(dt.isoformat()[11:13])+':'+str(dt.isoformat()[14:16])+':00')
    #wrfout = Dataset('/glade/derecho/scratch/mawilson/20210604_newcase/nature_100IOP4/final_nature/wrfout_d02_2022-'+str(dt.isoformat()[5:7])+'-'+str(dt.isoformat()[8:10])+'_'+str(dt.isoformat()[11:13])+':'+str(dt.isoformat()[14:16])+':00')
    #wrfout = Dataset('/glade/derecho/scratch/mawilson/20210604_newcase/nature_100JUNE/final_nature/wrfout_d02_2021-'+str(dt.isoformat()[5:7])+'-'+str(dt.isoformat()[8:10])+'_'+str(dt.isoformat()[11:13])+':'+str(dt.isoformat()[14:16])+':00')
    
    lon = wrfout.variables['XLONG']
    lat = wrfout.variables['XLAT']
    U10 = wrfout.variables['U10']
    V10 = wrfout.variables['U10']
    T2 = np.asarray(wrfout.variables['T2'])*units('K')
    T2F = T2 .to('degF')
    Q2 = np.asarray(wrfout.variables['Q2'])
    P2 = np.asarray(wrfout.variables['PSFC'][:]/100) * units('hPa')
    Td2 = dewpoint_from_specific_humidity(P2[0,:,:], T2[0,:,:], Q2[0,:,:]*units('kg/kg'))
    RH2 = relative_humidity_from_specific_humidity(P2[0,:,:], T2[0,:,:], Q2[0,:,:]*units('kg/kg'))
    SPD10 = wind_speed(np.asarray(U10)*units('m/s'), np.asarray(V10)*units('m/s'))
    cloud=wrfout.variables['QCLOUD']
    T_z = np.asarray(getvar(wrfout, "tk"))
    p_z = np.asarray(getvar(wrfout, "pres"))
    q_zi = np.asarray(wrfout.variables['QVAPOR'][:])
    q_z = specific_humidity_from_mixing_ratio(q_zi)
    u_z, v_z = getvar(wrfout, 'uvmet')
    u_z = np.asarray(u_z)
    v_z = np.asarray(v_z)
    z_z = np.asarray(getvar(wrfout, "height_agl"))
    #z_z = np.asarray(getvar(wrfout, "height"))
    z_zm = np.asarray(getvar(wrfout, "height"))
    td_z = dewpoint_from_specific_humidity(p_z*units('Pa'), T_z*units('K'), q_z*units('kg/kg')).to('K')
    
    #Convert WRF file time into same units as the obs_seq time
    dt_tot = (dt - datetime(1601,1,1)).total_seconds() / 86400
    time_diff = np.abs(dt_tot - uas_times2)
    
    #Get obs within +- 2.5 minutes of each WRF file
    time_T3 = uas_times2[time_diff<(150/86400)]
    #loc_T3 = loc_T2[time_diff<(150/86400), :]
    #qc_T3 = qc_T2[time_diff<(150/86400)]
    elev_T3 = uas_z2[time_diff<(150/86400)]
    if len(time_T3)==0:
            print('NO OBS IN WINDOW')
            continue
    #Get the correct lat/lon for an example point
    for st1 in range(len(prof_lats)):
        latp=prof_lats[st1]
        lonp=prof_lons[st1]
        #esfcp = prof_esfc[st1]
        #Get location for each ob in model land
        lon1d = np.ndarray.flatten(lon[0,:,:])
        lat1d = np.ndarray.flatten(lat[0,:,:])
        station = []
        points = []
        for i in range(len(lon1d)):
            points.append((lat1d[i],lon1d[i]))
            station.append((latp,lonp))
        dist = haversine.haversine_vector(station,points)
        dist2=dist.reshape(lon.shape[1],lon.shape[2])
        print(lon[0,:,:][np.where(dist2==np.min(dist2))])
        print(lat[0,:,:][np.where(dist2==np.min(dist2))])
        print(np.where(dist2==np.min(dist2)))
        st_xind = np.where(dist2==np.min(dist2))[0][0]
        st_yind = np.where(dist2==np.min(dist2))[1][0]

        #Actually get those interpolated obs
        # z_point = z_z[:,st_xind,st_yind]
        # t_point = T_z[:,st_xind,st_yind]
        # q_point = td_z[0,:,st_xind,st_yind].magnitude
        # u_point = u_z[:,st_xind,st_yind]
        # v_point = v_z[:,st_xind,st_yind]
        z_point = np.concatenate([[0], z_z[:,st_xind,st_yind]])
        t_point = np.concatenate([[T2[0,st_xind,st_yind].magnitude], T_z[:,st_xind,st_yind]])
        q_point = np.concatenate([[Td2[st_xind,st_yind].magnitude], td_z[0,:,st_xind,st_yind].magnitude])
        #q_point = np.concatenate([[Q2[0,st_xind,st_yind]], q_z[0,:,st_xind,st_yind]])
        u_point = np.concatenate([[U10[0,st_xind,st_yind]], u_z[:,st_xind,st_yind]])
        v_point = np.concatenate([[V10[0,st_xind,st_yind]], v_z[:,st_xind,st_yind]])
        zm_point = z_zm[:,st_xind,st_yind]
        esfc = zm_point[0]-z_point[0]
        p_point = np.concatenate([[P2[0,st_xind, st_yind].magnitude], p_z[:,st_xind,st_yind]/100])
        
        m = 0
        for point in elev_T3:
            
            T2_a = interpolate_1d(point, z_point, t_point)
            q2_a = interpolate_1d(point, z_point, q_point)
            u2_a = interpolate_1d(point, z_point, u_point)
            v2_a = interpolate_1d(point, z_point, v_point)
            p2_a = interpolate_1d(point, z_point, p_point)

            wspd_a = np.sqrt(u2_a**2 + v2_a**2)
            if wspd_a > wspd_thresh:
                print('wind is too strong, skipping')
            else:
            
                # error = np.random.normal(loc=0.0, scale=np.sqrt(oerr_T))
                # if np.abs(error/4) > (np.sqrt(oerr_T)*1.0):
                #     error = (error / np.abs(error)) * (np.sqrt(oerr_T)*1.0)
                # T2_b = T2_a + error/4
                # #uas_T.append(T2_b)
                # print(T2_b)
                # obs_s.append(T2_b)
                # otype_s.append(otype_T)
                # time_s.append(time_T3[m])
                # elev_s.append(point+esfc)
                # error_s.append(oerr_T)
                # lon_s.append(lonp)
                # lat_s.append(latp)
                # pres_s.append(p2_a)
                #Q section
                error = np.random.normal(loc=0.0, scale=np.sqrt(oerr_q))
                if np.abs(error/4) > (np.sqrt(oerr_q)*1.0):
                    error = (error / np.abs(error)) * (np.sqrt(oerr_q)*1.0)
                q2_b = q2_a + error/4
                #uas_q.append(q2_b)
                obs_s.append(q2_b)
                otype_s.append(otype_q)
                time_s.append(time_T3[m])
                elev_s.append(point+esfc)
                error_s.append(oerr_q*4)
                lon_s.append(lonp)
                lat_s.append(latp)
                pres_s.append(p2_a)
                
                # # #U section
                # error = np.random.normal(loc=0.0, scale=np.sqrt(oerr_u))
                # if np.abs(error/4) > (np.sqrt(oerr_u)*1.0):
                #     error = (error / np.abs(error)) * (np.sqrt(oerr_u)*1.0)
                # u2_b = u2_a + error/4
                # #uas_u.append(u2_b)
                # obs_s.append(u2_b)
                # otype_s.append(otype_u)
                # time_s.append(time_T3[m])
                # elev_s.append(point+esfc)
                # error_s.append(oerr_u)
                # lon_s.append(lonp)
                # lat_s.append(latp)
                # pres_s.append(p2_a)
                
                # # #V section
                # error = np.random.normal(loc=0.0, scale=np.sqrt(oerr_v))
                # if np.abs(error/4) > (np.sqrt(oerr_v)*1.0):
                #     error = (error / np.abs(error)) * (np.sqrt(oerr_v)*1.0)
                # v2_b = v2_a + error/4
                # #uas_v.append(v2_b)
                # obs_s.append(v2_b)
                # otype_s.append(otype_v)
                # time_s.append(time_T3[m])
                # elev_s.append(point+esfc)
                # error_s.append(oerr_v)
                # lon_s.append(lonp)
                # lat_s.append(latp)
                # pres_s.append(p2_a)
            
            m=m+1
    

2022-09-15 03:00:00
[-84.54797]
[39.211533]
(array([542]), array([685]))
[293.4418944]
[294.76974603]
[295.09558282]
[294.505333]
[294.16085735]
[294.31175051]
[293.88250925]
[293.9041394]
[293.71746704]
2022-09-15 03:05:00
[-84.54797]
[39.211533]
(array([542]), array([685]))
[293.74706978]
[293.25584644]
[292.91143051]
[292.60927241]
[292.59355817]
[291.73089647]
[291.83977499]
[291.24274333]
[290.66561731]
[290.46610423]
[290.01307778]
[289.73460676]


In [5]:
#Convert lats and lons to radians for DART, because why not
lon_DART = np.radians(np.asarray(lon_s))
lat_DART = np.radians(np.asarray(lat_s))

lon_DART = np.where(lon_DART > 0.0, lon_DART, lon_DART+(2.0*np.pi))

#Convert time into DART format. This is hacky now, improve later
day_DART = 154024
#day_DART = 153966
#day_DART = 153556

seconds_DART = (np.asarray(time_s) - day_DART) * 86400

In [6]:
for ob in obs_s:
    print(ob[0])

293.44189439912884
294.7697460290564
295.09558282058146
294.50533299982965
294.1608573456126
294.3117505096133
293.8825092539199
293.9041393983894
293.71746704257606
293.7470697821908
293.25584643645493
292.9114305081519
292.60927240665785
292.5935581675143
291.73089647486455
291.83977498533085
291.2427433266425
290.6656173059426
290.4661042290728
290.01307777546214
289.7346067557669


In [7]:
#Sort everything in time order
inds_time = seconds_DART.argsort()
# print(seconds_DART)
# print(seconds_DART[inds_time])
seconds_DART1 = seconds_DART[inds_time]
seconds_DART1[seconds_DART1 < 0] = 0
obs_s1 = np.asarray(obs_s)[inds_time]
lon_DART1 = lon_DART[inds_time]
lat_DART1 = lat_DART[inds_time]
elev_s1 = np.asarray(elev_s)[inds_time]
otype_s1 = np.asarray(otype_s)[inds_time]
error_s1 = np.asarray(error_s)[inds_time]
pres_s1 = np.asarray(pres_s)[inds_time]

In [8]:
print(pres_s1)

[[989.39727783]
 [983.84753008]
 [978.22298949]
 [972.59372458]
 [966.98747941]
 [961.42312322]
 [955.88074298]
 [950.37251394]
 [944.88250854]
 [940.91149769]
 [935.49238286]
 [930.0877599 ]
 [924.69736932]
 [919.34992239]
 [914.01284043]
 [908.68911398]
 [903.39330144]
 [898.1240848 ]
 [892.87711875]
 [887.66881414]
 [882.46050953]]


In [9]:
for bigfoot in [1,2]:
    print(bigfoot)
    #Write the simulated obs out to an obs_seq file
    filename = 'SIM_UAS_IOP6_TD_grid'
    fi = open(filename, "w")
    fi.write(" obs_sequence\n")
    fi.write("obs_kind_definitions\n")
    fi.write("    %d \n" %(4))
    fi.write("    %d          %s   \n" %(123, 'UAS_TEMPERATURE'))
    fi.write("    %d          %s   \n" %(120, 'UAS_DEWPOINT'))
    #fi.write("    %d          %s   \n" %(124, 'UAS_SPECIFIC_HUMIDITY'))
    fi.write("    %d          %s   \n" %(121, 'UAS_U_WIND_COMPONENT'))
    fi.write("    %d          %s   \n" %(122, 'UAS_V_WIND_COMPONENT'))
    
    fi.write("  num_copies:            %d  num_qc:            %d\n" % (1,1))
    fi.write(" num_obs:       %d  max_num_obs:       %d\n" % (len(obs_s1), len(obs_s1)))
    fi.write("MADIS observation\n")
    fi.write("Data QC\n")
    
    fi.write("  first:            %d  last:       %d\n" % (1, len(obs_s1)))
    
    for q in range(len(obs_s)):
    
        fi.write(" OBS            %d\n" % (q+1) )
        fi.write("   %20.14f\n" % obs_s1[q] )
        fi.write("   %20.14f\n" % 1.0 )
    
        if q+1 == 1:
            fi.write(" %d %d %d\n" % (-1, q+2, -1) ) #First obs
        elif q+1 == len(obs_s):
            fi.write(" %d %d %d\n" % (q, -1, -1) ) #Last obs
        else:
            fi.write(" %d %d %d\n" % (q, q+2, -1) )
    
        fi.write("obdef\n")
        fi.write("loc3d\n")
        fi.write("    %20.14f          %20.14f          %20.14f     %d\n" %
                           (lon_DART1[q], lat_DART1[q], pres_s1[q]*100, 2))
        fi.write("kind\n")
    
        fi.write("     %d     \n" % otype_s1[q])
    
        fi.write("    %d          %d     \n" % (seconds_DART1[q], day_DART))
    
        fi.write("    %20.14f  \n" % error_s1[q])

1
2


/glade/derecho/scratch/mawilson/tmp/ipykernel_30147/1876131446.py:25: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  fi.write("   %20.14f\n" % obs_s1[q] )
/glade/derecho/scratch/mawilson/tmp/ipykernel_30147/1876131446.py:37: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  fi.write("    %20.14f          %20.14f          %20.14f     %d\n" %


In [10]:
print((st_sec/86400))

0.125


In [11]:
print(fl_num*(st_sec/86400))

1.25


In [12]:
print(np.asarray(obs_s1))
print(np.asarray(elev_s1))

[[293.4418944 ]
 [294.76974603]
 [295.09558282]
 [294.505333  ]
 [294.16085735]
 [294.31175051]
 [293.88250925]
 [293.9041394 ]
 [293.71746704]
 [293.74706978]
 [293.25584644]
 [292.91143051]
 [292.60927241]
 [292.59355817]
 [291.73089647]
 [291.83977499]
 [291.24274333]
 [290.66561731]
 [290.46610423]
 [290.01307778]
 [289.73460676]]
[ 279.51815796  329.51815796  379.51815796  429.51815796  479.51815796
  529.51815796  579.51815796  629.51815796  679.51815796  729.5241394
  779.5241394   829.5241394   879.5241394   929.5241394   979.5241394
 1029.5241394  1079.5241394  1129.5241394  1179.5241394  1229.5241394
 1279.5241394 ]
